# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riteshy1526/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — Search and content performance signals can identify content opportunities

The research describes relationships between search performance and content opportunities. My methodology question is: **where exactly does the label or outcome used for this finding come from?** If the label is created from observed search or traffic outcomes, I would want to know the time window used to create it and whether those outcomes were available only after the prediction point.

This matters because a label created from future information can make a model or analysis look stronger than it would be in a real decision setting. I would therefore check that the label is clearly separated from the features available at prediction time.

### Finding 2 — Machine-learning signals can support better content decisions

The research suggests that data-driven signals can help identify useful content actions. My methodology question is: **does the validation design support the strength of this claim?** In particular, I would check whether the evaluation split prevents related pages from the same client or time period from appearing across both training and evaluation data.

A grouped or time-aware validation design would provide stronger evidence that the result can generalize beyond the specific pages used during training. I see this as a constructive check rather than a criticism of the research.

In [1]:
# Section 1: simple completeness check

paper_findings = {
    "finding_1": {
        "finding_present": True,
        "label_question_present": True,
        "validation_question_present": True
    },
    "finding_2": {
        "finding_present": True,
        "label_question_present": True,
        "validation_question_present": True
    }
}

for name, checks in paper_findings.items():
    print(name, ":", all(checks.values()))

print("\nSection 1 completed.")

finding_1 : True
finding_2 : True

Section 1 completed.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I first reproduce the Week-5 data preparation and evaluate the model using the original validation approach. I then use a client-grouped split so that pages from the same client do not appear in both training and validation sets.

The purpose of the comparison is not to claim that one split is universally correct. It is to measure how sensitive the observed model performance is to a more conservative validation design.

In [6]:
# Look for columns that may represent the Week-5 target or ranking score

possible_target_columns = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in ["score", "target", "label", "opportunity", "refresh"]
    )
]

print("Possible target columns:")
for col in possible_target_columns:
    print("-", col)

Possible target columns:


In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


### Week-5 feature and target preparation

I reuse the same measurable signals used in my Week-5 model rather than creating a new modeling task. The purpose of this audit is to test whether the observed Week-5 result remains similar under a more conservative validation design.

In [5]:
# Inspect the main fields used by the Week-5 refresh-ranking approach

candidate_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct"
]

features = [col for col in candidate_features if col in df.columns]

print("Features available:", len(features))
print(features)

Features available: 29
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct']


In [9]:
TARGET = "refresh_score"
print(TARGET)

refresh_score


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I reviewed the final feature set for information that may not have been available when the content-refresh decision would have been made.

The main timing question is whether each feature describes information available before the prediction point or contains information from a future evaluation period. Historical traffic, search-position, CTR, and trend features are useful signals, but their measurement windows need to end before the decision point.

I therefore treat timing as an explicit validation requirement rather than assuming that a feature is safe because its name sounds historical.I reviewed the final feature set for information that may not have been available when the content-refresh decision would have been made.

The main timing question is whether each feature describes information available before the prediction point or contains information from a future evaluation period. Historical traffic, search-position, CTR, and trend features are useful signals, but their measurement windows need to end before the decision point.

I therefore treat timing as an explicit validation requirement rather than assuming that a feature is safe because its name sounds historical.

In [10]:
# Leakage audit for the final Week-5 feature set

leakage_keywords = [
    "impressions",
    "clicks",
    "sessions",
    "ctr",
    "position",
    "trend",
    "engagement",
    "scroll",
    "ai_traffic"
]

audit_rows = []

for feature in features:
    lower_feature = feature.lower()

    if any(keyword in lower_feature for keyword in leakage_keywords):
        risk = "Timing check required"
        reason = "Performance-derived feature; verify that its measurement window ends before prediction."
    else:
        risk = "Lower timing risk"
        reason = "No obvious future-performance signal from the feature name; availability should still be confirmed."

    audit_rows.append({
        "feature": feature,
        "leakage_status": risk,
        "reason": reason
    })

leakage_audit = pd.DataFrame(audit_rows)

leakage_audit

,feature,leakage_status,reason
0,search_volume,Lower timing risk,No obvious future-performance signal from the ...
1,competition,Lower timing risk,No obvious future-performance signal from the ...
2,cpc,Lower timing risk,No obvious future-performance signal from the ...
3,word_count,Lower timing risk,No obvious future-performance signal from the ...
4,char_count,Lower timing risk,No obvious future-performance signal from the ...
5,impressions_90d,Timing check required,Performance-derived feature; verify that its m...
6,clicks_90d,Timing check required,Performance-derived feature; verify that its m...
7,pageviews_90d,Lower timing risk,No obvious future-performance signal from the ...
8,sessions_90d,Timing check required,Performance-derived feature; verify that its m...
9,users_90d,Lower timing risk,No obvious future-performance signal from the ...


In [11]:
print("Total features audited:", len(leakage_audit))
print(
    "Features requiring timing checks:",
    (leakage_audit["leakage_status"] == "Timing check required").sum()
)

Total features audited: 29
Features requiring timing checks: 20


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original claim

A strong interpretation of my Week-5 work could be:

> "The model predicts which content should be refreshed."

This claim goes beyond what the evaluation directly measures. The model was evaluated using historical data, so the result does not by itself prove that following the ranking will improve future traffic or guarantee that a page should be refreshed.

### Safer claim

> "The model provides an observed and measured directional ranking signal that can support content-refresh prioritization on the evaluated dataset."

This version is more appropriate because it describes what was measured and presents the model as decision-support rather than as proof of a future business outcome.

In [12]:
# Check that the rewritten claim uses evidence-safe language

safe_terms = [
    "observed",
    "measured",
    "directional",
    "support"
]

safe_claim = (
    "The model provides an observed and measured directional ranking "
    "signal that can support content-refresh prioritization on the evaluated dataset."
)

print("Rewritten claim:")
print(safe_claim)

print("\nEvidence-safe terms found:")
for term in safe_terms:
    print(f"{term}: {term.lower() in safe_claim.lower()}")

Rewritten claim:
The model provides an observed and measured directional ranking signal that can support content-refresh prioritization on the evaluated dataset.

Evidence-safe terms found:
observed: True
measured: True
directional: True
support: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.